# Missing data and time patterns — Code-Along Template

**Type into this notebook during class.** The explanation is here; the code is not.

| | |
|---|---|
| **Follow along** | Write the code in each `# TODO` cell. The hint names the functions and arguments. |
| **If you fall behind** | Do not copy frantically. Skip to the next **CHECKPOINT**, run it, rejoin. |
| **If a cell errors** | Later cells depend on it and will fail too. Run the next CHECKPOINT rather than debugging under time pressure. |
| **After class** | The worked version is `D005_02b_Missing_Data.ipynb`. Compare, do not copy. |

Cells that are already filled in are deliberate — imports and quick glances at the
data cost class time and teach nothing.

# C2: Addressing Missing Data in Python

##  Learning Objectives

By the end of this notebook, you will be able to:
- Understand the different types of missing data (MCAR, MAR, MNAR)
- Visualize missing data patterns effectively
- Implement strategies for handling missing data:
  - Deletion methods
  - Simple imputation (Mean, Median, Mode)
  - Advanced imputation (KNN)
- Apply these techniques to a real-world dataset

---

## 1. Introduction to Missing Data

Missing data is a common problem in real-world datasets. Understanding *why* data is missing is crucial for choosing the right handling strategy.

### Types of Missing Data

1. **Missing Completely At Random (MCAR)**:
   - The missingness is unrelated to the data.
   - Example: A data entry error or equipment failure.

2. **Missing At Random (MAR)**:
   - The missingness is related to some observed data but not the missing value itself.
   - Example: Women might be less likely to report their age, but this propensity depends on their gender (which is observed), not their actual age.

3. **Missing Not At Random (MNAR)**:
   - The missingness is related to the missing value itself.
   - Example: People with very high incomes might be less likely to disclose their income.

### Visual Summary of Missing Data Types

![Missing Data Simulation under MCAR, MAR, MNAR](https://bookdown.org/rwnahhas/RMPH/Images/MissingDataSimulation.png)
*Source: Introduction to Regression Methods for Public Health Using R - Bias comparison under different missing data mechanisms*

| Mechanism | Definition | Example | Impact on Analysis |
|-----------|------------|---------|-------------------|
| **MCAR** | Missingness unrelated to any data | Random sensor failure | Least bias, safe to delete |
| **MAR** | Missingness related to observed data | Older people skip online surveys | Moderate bias, use imputation |
| **MNAR** | Missingness related to the missing value itself | High earners hide income | Most bias, requires modeling |

---

## 2. Environmental Setup

We will use `pandas` for data manipulation, `matplotlib` and `seaborn` for visualization, and `sklearn` for advanced imputation.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Advanced imputation
from sklearn.impute import KNNImputer

# Settings
plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', None)

print("Libraries imported successfully!")

Libraries imported successfully!


---

## 3. Data Loading and Initial Inspection

We will continue using the **Divorce Data Analysis** dataset.

### Load the Data

In [ ]:
# TODO: compute `df`
# Hint: pd.read_csv, df.head
# Name your result: df

In [ ]:
# ═══ CHECKPOINT — "Identify Missing Values" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
df = pd.read_csv('../../00-data/Divorce Data Analysis.csv')

print("Checkpoint restored. Ready for: Identify Missing Values")

### Identify Missing Values

Let's see the count and percentage of missing values for each column.

In [ ]:
# TODO: compute `missing_counts`, `missing_percentages`, `missing_df`
# Hint: .sum(), .sort_values(), df.isnull, pd.DataFrame
# Name your result: missing_counts, missing_percentages, missing_df

---

## 4. Visualizing Missing Data

Visualizing missing data patterns can help identify if data is MCAR, MAR, or MNAR.

### Heatmap of Missing Values

A heatmap shows the presence (or absence) of missing values across the dataset.

In [ ]:
# TODO: draw a `plt.figure`
# Hint: plt.figure, sns.heatmap, plt.title, plt.show, df.isnull

**Observation**: Look for horizontal lines (missing in one row across multiple columns) or vertical blocks. Random scattered yellow lines suggest randomness.

---

## 5. Methods for Handling Missing Data

### Method 1: Deletion

**Listwise Deletion** removes the entire row if any data is missing. 

- **Pros**: Simple, preserves distribution (if MCAR).
- **Cons**: Reduces sample size, can introduce bias if not MCAR.

In [5]:
# Copy original dataframe
df_clean = df.copy()

# Drop rows with any missing values
df_dropped = df_clean.dropna()

print(f"Original rows: {len(df)}")
print(f"Rows after dropping NA: {len(df_dropped)}")
print(f"Rows lost: {len(df) - len(df_dropped)} ({((len(df) - len(df_dropped))/len(df))*100:.2f}%)")

Original rows: 2209
Rows after dropping NA: 1330
Rows lost: 879 (39.79%)


### Method 2: Simple Imputation (Mean/Median/Mode)

Instead of deleting, we can fill missing values with estimated ones.

#### 2.1 Numerical Data: Mean or Median Imputation
- **Mean**: Good for normally distributed data.
- **Median**: Better for skewed data (outliers).

In [6]:
# Example: Impute 'income_man' with Median
df_imputed = df.copy()

# Check original statistics
print("Original Mean Income (Man):", df['income_man'].mean())
print("Original Median Income (Man):", df['income_man'].median())

# Impute with median
median_income = df['income_man'].median()
df_imputed['income_man'] = df_imputed['income_man'].fillna(median_income)

print("Missing count after imputation:", df_imputed['income_man'].isnull().sum())

Original Mean Income (Man): 7922.9864101403355
Original Median Income (Man): 6000.0
Missing count after imputation: 0


#### 2.2 Categorical Data: Mode Imputation

For categorical columns, we fill with the most frequent value (Mode).

In [7]:
# Example: Impute 'education_man' with Mode
mode_education = df['education_man'].mode()[0]
print(f"Most frequent education level: {mode_education}")

df_imputed['education_man'] = df_imputed['education_man'].fillna(mode_education)
print("Missing count after imputation:", df_imputed['education_man'].isnull().sum())

Most frequent education level: Professional
Missing count after imputation: 0


In [ ]:
# ═══ CHECKPOINT — "Method 3: Advanced Imputation (KNN)" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
df = pd.read_csv('../../00-data/Divorce Data Analysis.csv')
missing_counts = df.isnull().sum()
missing_percentages = df.isnull().sum() / len(df) * 100
missing_df = pd.DataFrame({'Missing Counts': missing_counts, 'Percentage (%)': missing_percentages}).sort_values(by='Percentage (%)', ascending=False)
df_clean = df.copy()
df_dropped = df_clean.dropna()
df_imputed = df.copy()
median_income = df['income_man'].median()
df_imputed['income_man'] = df_imputed['income_man'].fillna(median_income)
mode_education = df['education_man'].mode()[0]
df_imputed['education_man'] = df_imputed['education_man'].fillna(mode_education)

print("Checkpoint restored. Ready for: Method 3: Advanced Imputation (KNN)")

### Method 3: Advanced Imputation (KNN)

**K-Nearest Neighbors (KNN) Imputation**:
- Finds 'k' rows that are most similar to the row with missing data.
- Averages the values of those neighbors to fill the missing spot.
- **Pros**: More accurate than simple mean/median.
- **Cons**: Computationally expensive for large datasets.

In [8]:
# KNN Imputation only works on numerical data
numerical_cols = ['income_man', 'income_woman', 'marriage_duration', 'num_kids']

# Initialize KNN Imputer
imputer = KNNImputer(n_neighbors=5)

# Perform imputation
df_knn = df.copy()
df_knn[numerical_cols] = imputer.fit_transform(df[numerical_cols])

print("Data after KNN Imputation (check first 5 rows):")
print(df_knn[numerical_cols].head())

Data after KNN Imputation (check first 5 rows):
   income_man  income_woman  marriage_duration  num_kids
0      2000.0        1800.0                5.0       1.0
1      6000.0        6000.0                7.0       1.4
2      5000.0        5000.0                2.0       2.0
3     12000.0       12000.0                2.0       2.0
4      6000.0       10000.0                3.0       1.0


---

## 6. Comparing Methods

Let's visualize how different imputation methods affect the distribution of the data.

In [ ]:
# TODO: draw a `plt.figure` of `income_man`
# Hint: plt.figure, sns.kdeplot, df.copy, .fillna(), plt.title
# Name your result: df_mean_imp, df_median_imp

**Insight:** Notice how Mean/Median imputation can distort the distribution by creating a high peak at the imputed value. KNN usually preserves the distribution better.

---

## 7.  Exercises

### Exercise 1: Investigate 'num_kids'
1. Calculate the percentage of missing values in `num_kids`.
2. Visualize the distribution of `num_kids`.
3. Choose an appropriate imputation method (Mean, Median, or Mode) and explain why.

### Decision Framework: Choosing an Imputation Strategy

Use this framework to decide which imputation method to apply:

```
                    Is missing % > 50?
                          |
              +-----------+-----------+
              | YES                   | NO
              v                       v
        Drop column           Is missing % < 5?
                                     |
                         +-----------+-----------+
                         | YES                   | NO
                         v                       v
                   Listwise Delete        Is data categorical?
                   or Simple Impute              |
                                     +-----------+-----------+
                                     | YES                   | NO
                                     v                       v
                                Mode Impute           Is data skewed?
                                                            |
                                                +-----------+-----------+
                                                | YES                   | NO
                                                v                       v
                                          Median Impute          Mean Impute
                                          or KNN                 or KNN
```

**Rule of Thumb:**
- Missing < 5%: Safe to delete or simple imputation
- Missing 5-50%: Use imputation (choose based on data type and distribution)
- Missing > 50%: Consider dropping the column entirely

In [10]:
# Your code here


### Exercise 2: Handling 'marriage_duration'
Use KNN Imputation to fill missing values in `marriage_duration`. Compare the mean and standard deviation before and after imputation.

In [ ]:
# Your code here


In [ ]:
# ═══ CHECKPOINT — "Validating Imputation Quality" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
df = pd.read_csv('../../00-data/Divorce Data Analysis.csv')
missing_counts = df.isnull().sum()
missing_percentages = df.isnull().sum() / len(df) * 100
missing_df = pd.DataFrame({'Missing Counts': missing_counts, 'Percentage (%)': missing_percentages}).sort_values(by='Percentage (%)', ascending=False)
df_clean = df.copy()
df_dropped = df_clean.dropna()
df_imputed = df.copy()
median_income = df['income_man'].median()
df_imputed['income_man'] = df_imputed['income_man'].fillna(median_income)
mode_education = df['education_man'].mode()[0]
df_imputed['education_man'] = df_imputed['education_man'].fillna(mode_education)
numerical_cols = ['income_man', 'income_woman', 'marriage_duration', 'num_kids']
imputer = KNNImputer(n_neighbors=5)
df_knn = df.copy()
df_knn[numerical_cols] = imputer.fit_transform(df[numerical_cols])
df_mean_imp = df.copy()
df_mean_imp['income_man'] = df['income_man'].fillna(df['income_man'].mean())
df_median_imp = df.copy()
df_median_imp['income_man'] = df['income_man'].fillna(df['income_man'].median())

print("Checkpoint restored. Ready for: Validating Imputation Quality")

### 6.1 Validating Imputation Quality

It's important to verify that imputation doesn't distort your data. Here's a function to compare distributions before and after:

In [ ]:
# TODO: compute `stats` from `Original`
# Hint: validate_imputation, pd.DataFrame, .round(), plt.subplots, .hist()
# Name your result: stats

### Exercise 3: Compare Imputation Methods
For the `num_kids` column:
1. Create three versions: mean-imputed, median-imputed, and KNN-imputed
2. Calculate the standard deviation for each version
3. Which method preserves the original distribution best? Why?

**Hint**: Use the `validate_imputation` function from Section 6.1 to compare.

In [ ]:
# Your code here

### Exercise 4: Real-World Scenario
You have a dataset where 15% of income data is missing. You suspect higher-income individuals were less likely to report their income.

1. What type of missing data is this (MCAR, MAR, or MNAR)?
2. Why might simple mean imputation be problematic here?
3. What alternative approach would you suggest?

**Discussion points to consider**: Selection bias, underestimation of variance, potential for biased estimates.

---

## 8. Conclusion

In this notebook, we learned:
- **MCAR, MAR, MNAR**: The types of missing data mechanisms.
- **Visualization**: Using heatmaps to spot missing patterns.
- **Strategies**: 
  - **Deletion** (Quick but loses data)
  - **Simple Imputation** (Easy but can distort distribution)
  - **KNN Imputation** (More accurate but computationally heavier)

Choosing the right method depends on your specific dataset and the amount of missing information.

**Next Steps**: Apply these cleaning techniques before moving to hypothesis testing or modeling.